In [1]:
# !pip install langchain_huggingface 
# !pip install langchain_community

In [2]:
import os
import faiss
import spacy
import re
import contractions
from textblob import TextBlob
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI

d:\Study\pyspider\GenAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\HP\AppData\Local\Temp\ipykernel_16656\319429306.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


### 1. Load the document(.txt)

In [3]:
data = open('data.txt', 'r').read()

### 2. Text Normalization

#### Converting text into lowercase

In [4]:
data = data.lower()

### removing numbers like 1., 2. 

In [5]:
data = re.sub('\d+\.',"", data)
data = re.sub(r"\b\d+(?:\.\d+)*\.?\b", "", data)
data

'====================================================================\nartificial intelligence (ai), machine learning (ml) and\ndeep learning (dl) - a complete guide\n====================================================================\n\ntable of contents\n------------------\n introduction\n what is artificial intelligence (ai)?\n history and evolution of ai\n types of artificial intelligence\n what is machine learning (ml)?\n how machine learning works\n types of machine learning\n    supervised learning\n    unsupervised learning\n    semi-supervised learning\n    reinforcement learning\n common machine learning algorithms\n what is deep learning (dl)?\n how deep learning works\n neural network architectures\n difference between ai, ml and dl\n applications of ai, ml and dl\n popular tools and frameworks\n challenges and limitations\n ethics and responsible ai\n future trends\n conclusion\n glossary of terms\n\n====================================================================\n i

#### expanding the words

In [6]:
data = contractions.fix(data)

#### removing punctuations and spl characters

In [7]:
data = re.sub('[^0-9a-zA-Z\s]', "", data).strip()
data

'artificial intelligence ai machine learning ml and\ndeep learning dl  a complete guide\n\n\ntable of contents\n\n introduction\n what is artificial intelligence ai\n history and evolution of ai\n types of artificial intelligence\n what is machine learning ml\n how machine learning works\n types of machine learning\n    supervised learning\n    unsupervised learning\n    semisupervised learning\n    reinforcement learning\n common machine learning algorithms\n what is deep learning dl\n how deep learning works\n neural network architectures\n difference between ai ml and dl\n applications of ai ml and dl\n popular tools and frameworks\n challenges and limitations\n ethics and responsible ai\n future trends\n conclusion\n glossary of terms\n\n\n introduction\n\n\nartificial intelligence machine learning and deep learning are three\nof the most talkedabout technologies of the twentyfirst century\nthey are reshaping industries changing how businesses operate and\ninfluencing everyday life

### removing extra spaces

In [8]:
data = re.sub("\s+", " ", data).strip()
data

'artificial intelligence ai machine learning ml and deep learning dl a complete guide table of contents introduction what is artificial intelligence ai history and evolution of ai types of artificial intelligence what is machine learning ml how machine learning works types of machine learning supervised learning unsupervised learning semisupervised learning reinforcement learning common machine learning algorithms what is deep learning dl how deep learning works neural network architectures difference between ai ml and dl applications of ai ml and dl popular tools and frameworks challenges and limitations ethics and responsible ai future trends conclusion glossary of terms introduction artificial intelligence machine learning and deep learning are three of the most talkedabout technologies of the twentyfirst century they are reshaping industries changing how businesses operate and influencing everyday life in ways most people do not even notice from voice assistants like siri and alexa

In [9]:
# corrected_words = TextBlob(data).correct()

In [10]:
# data = str(corrected_words)
# data

#### lematization, removing stopwords

In [11]:
nlp = spacy.load('en_core_web_sm')

In [12]:
tokens = nlp(data)
lemmatize_tokens = [token.lemma_ for token in tokens if not token.is_stop]
data = " ".join(lemmatize_tokens).strip()
data

'artificial intelligence ai machine learn ml deep learning dl complete guide table content introduction artificial intelligence ai history evolution ai type artificial intelligence machine learn ml machine learn work type machine learn supervise learning unsupervised learning semisupervise learn reinforcement learn common machine learning algorithm deep learning dl deep learning work neural network architecture difference ai ml dl application ai ml dl popular tool framework challenge limitation ethic responsible ai future trend conclusion glossary term introduction artificial intelligence machine learning deep learning talkedabout technology twentyfirst century reshape industry change business operate influence everyday life way people notice voice assistant like siri alexa recommendation system netflix amazon selfdrive car medical diagnosis tool technology term interchangeably casual conversation thing artificial intelligence broad concept machine learning subset ai deep learning subs

#### chunking

In [13]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, 
    chunk_overlap=40, 
    )
data = text_splitter.create_documents([data])
data

[Document(metadata={}, page_content='artificial intelligence ai machine learn ml deep learning dl complete guide table content introduction artificial intelligence ai history evolution ai type artificial intelligence machine learn ml'),
 Document(metadata={}, page_content='intelligence machine learn ml machine learn work type machine learn supervise learning unsupervised learning semisupervise learn reinforcement learn common machine learning algorithm deep learning dl'),
 Document(metadata={}, page_content='learning algorithm deep learning dl deep learning work neural network architecture difference ai ml dl application ai ml dl popular tool framework challenge limitation ethic responsible ai future'),
 Document(metadata={}, page_content='limitation ethic responsible ai future trend conclusion glossary term introduction artificial intelligence machine learning deep learning talkedabout technology twentyfirst century reshape industry'),
 Document(metadata={}, page_content='twentyfirst 

In [14]:
print(data[0])
print(type(data[0]))
print(data[0].page_content)

page_content='artificial intelligence ai machine learn ml deep learning dl complete guide table content introduction artificial intelligence ai history evolution ai type artificial intelligence machine learn ml'
<class 'langchain_core.documents.base.Document'>
artificial intelligence ai machine learn ml deep learning dl complete guide table content introduction artificial intelligence ai history evolution ai type artificial intelligence machine learn ml


In [15]:
print(data)
data[0].metadata={'file_name':'data.txt'}
print(data)

[Document(metadata={}, page_content='artificial intelligence ai machine learn ml deep learning dl complete guide table content introduction artificial intelligence ai history evolution ai type artificial intelligence machine learn ml'), Document(metadata={}, page_content='intelligence machine learn ml machine learn work type machine learn supervise learning unsupervised learning semisupervise learn reinforcement learn common machine learning algorithm deep learning dl'), Document(metadata={}, page_content='learning algorithm deep learning dl deep learning work neural network architecture difference ai ml dl application ai ml dl popular tool framework challenge limitation ethic responsible ai future'), Document(metadata={}, page_content='limitation ethic responsible ai future trend conclusion glossary term introduction artificial intelligence machine learning deep learning talkedabout technology twentyfirst century reshape industry'), Document(metadata={}, page_content='twentyfirst cent

#### chunk Embeddings

In [16]:
embeddings_model = HuggingFaceEmbeddings(
    model_name = 'sentence-transformers/all-miniLM-L6-V2'
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1536.74it/s]


In [17]:
vectordb = FAISS.from_documents(documents=data, embedding=embeddings_model)
vectordb

In [18]:
user_query = 'what is Machine Learning ?'
ret_chunks = vectordb.similarity_search(user_query)


In [19]:
updated_ret_chunks = set()
for chunk in ret_chunks:
    updated_ret_chunks.add(chunk.page_content)
updated_ret_chunks
r_text = "\n".join(updated_ret_chunks)
r_text

'machine learning supervise learning supervise learning involve train model label dataset mean training example include input datum correct output goal model learn mapping function predict output new\nintelligence machine learn ml machine learn work type machine learn supervise learning unsupervised learning semisupervise learn reinforcement learn common machine learning algorithm deep learning dl\nmachine learning workflow involve follow step step datum collection gather relevant datum source database apis sensor web step datum preprocesse cleaning prepare datum analysis include handle miss\npopular supervised learning algorithm include linear regression logistic regression decision tree random forest support vector machine gradient boost method like xgboost lightgbm unsupervised learn'

In [ ]:
def rag_query(query, k=2):
    # faiss.normalize_L2(query)
    r_chunks = vectordb.similarity_search(query)
    # print(r_chunks)
    r_chunks = [doc.page_content for doc in r_chunks]
    r_string = " ".join(r_chunks)
    
    prompt = f'''
              You're a helpful assistant
              Assigned Task for you : structure my output => {r_string}
              for this input -> {query}
              note :
               1) don't add extra contents just structure mentioned output
               2) if there is mistake in output correct or else keep the original output
               with structures result.
               output structure:
               Input : {user_prompt}
               output : structured output

    '''
    llm_model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash", # gemini-2.5-flash gemini-3.6-flash gemini-2.5-pro
    # api_key = os.environ['GEMINI_API_KEY']
    )
    response = llm_model.invoke(prompt)
    return response
user_prompt = "Explain GenAi ?"
user_prompt = re.sub('[^0-9a-zA-Z]', "", user_prompt)

response = rag_query(user_prompt)
print(response)

content=[{'type': 'text', 'text': 'Input : Explain GenAi ?\noutput :\n\n### **1. The Generative AI Era (2020s)**\n* **Emergence of LLMs:** The 2020s marked the rise of Large Language Models (LLMs) and generative AI tools capable of producing human-like text, images, audio, and video.\n* **Mainstream Adoption:** This era marks a new chapter in AI history, bringing AI into mainstream use for millions of people.\n* **Responsible Deployment:** A key focus is to develop and deploy these AI technologies responsibly.\n\n### **2. Core Capabilities and AI Systems**\n* **AI Agents and Autonomous Systems:** Systems capable of performing complex, multi-step tasks with minimal human intervention.\n* **Explainable AI (XAI):** Development techniques aimed at making AI decision-making transparent and interpretable.\n* **AI for Scientific Discovery:** Using AI to assist and accelerate scientific breakthroughs.\n\n### **3. Future Trends in Generative AI**\n* **Advanced Generative Models:** Continued adv

In [25]:
response.text

'Here is the structured and corrected version of your output, organized logically with improved grammar, punctuation, and formatting while preserving all of your original content:\n\n***\n\n### 1. AI Agents and Autonomous Systems\n* **Definition:** Systems capable of performing complex, multi-step tasks with minimal human intervention.\n\n### 2. Explainable AI (XAI)\n* **Definition:** Development techniques designed to make AI decision-making transparent and interpretable.\n\n### 3. AI in Scientific Discovery\n* **Application:** Utilizing artificial intelligence to advance and accelerate scientific discovery.\n\n### 4. The Generative AI Era (2020s)\n* **The Emergence:** Marked by the rise of Large Language Models (LLMs) and generative AI tools capable of producing human-like text, images, audio, and video. \n* **Impact:** Marks a new chapter in AI history, bringing AI into mainstream use for millions of people.\n* **Requirement:** The critical need to develop and deploy AI responsibly.